# 00. セットアップ — 環境構築・認証・パス設定

このノートブックでは以下を行います:

1. **Google Drive マウント** — Colab 実行時のみ
2. **GitHub リポジトリ取得 / 最新化** — `git clone` または `git pull`
3. **pip install** — 必要パッケージの一括インストール
4. **環境変数チェック** — API キーの確認
5. **パス設定** — `staging/` サブディレクトリの確保

このノートブックを最初に実行してください。以降のノートブック (`01_collect_raw_data.ipynb` 等) はここで定義した変数を前提とします。

In [ ]:
# ── Google Drive マウント（Colab 実行時のみ）──────────────────────────────────
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive/AI_TradeManagement"
else:
    DRIVE_BASE = None  # ローカル実行時は不要

print(f"IN_COLAB={IN_COLAB}, DRIVE_BASE={DRIVE_BASE}")

In [ ]:
# ── GitHub からリポジトリ取得 / 最新化 ──────────────────────────────────────
# Private リポジトリの場合: Colab Secrets に GITHUB_TOKEN を登録してください
#   GitHub → Settings → Developer settings → Personal access tokens (classic)
#   → repo スコープで生成 → Colab「🔑シークレット」に GITHUB_TOKEN として追加

REPO_NAME = "AI_TradeManagement"
REPO_OWNER = "tsp0918"  # GitHubユーザー名

if IN_COLAB:
    import subprocess

    # GitHub Token を Secrets から取得
    _github_token = ""
    try:
        from google.colab import userdata
        _github_token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        import os
        _github_token = os.environ.get("GITHUB_TOKEN", "")

    if not _github_token:
        print("⚠️  GITHUB_TOKEN が未設定です。")
        print("   手順: GitHub → Settings → Developer settings → Personal access tokens")
        print("         → repo スコープで生成 → Colab シークレット「GITHUB_TOKEN」に登録")
        raise RuntimeError("GITHUB_TOKEN が必要です。上記手順で設定してください。")

    REPO_URL = f"https://{_github_token}@github.com/{REPO_OWNER}/{REPO_NAME}"

    # clone または pull
    result = subprocess.run(
        f"git -C /content/{REPO_NAME} pull || git clone {REPO_URL} /content/{REPO_NAME}",
        shell=True, capture_output=True, text=True
    )
    output = result.stdout or result.stderr
    # トークンをログから除去して表示
    print(output.replace(_github_token, "***"))

    REPO_BASE = f"/content/{REPO_NAME}"
else:
    import pathlib
    REPO_BASE = str(pathlib.Path("/Users/takehirosato/Desktop/AI_TradeManagement"))

print(f"REPO_BASE={REPO_BASE}")
import sys
sys.path.insert(0, f"{REPO_BASE}/scripts")

In [ ]:
# ── 依存パッケージのインストール ──────────────────────────────────────────────
!pip install -q requests anthropic sentence-transformers faiss-cpu

In [ ]:
# ── 環境変数の設定 ────────────────────────────────────────────────────────────
# Google Colab: 左サイドバー「🔑 シークレット」→ ANTHROPIC_API_KEY を追加 →
#               「ノートブックのアクセス」トグルを ON → このセルを再実行
import os

ANTHROPIC_API_KEY = ""
BIS_API_KEY       = "DEMO_KEY"

# ① Colab Secrets から取得
_colab_secrets_ok = False
try:
    from google.colab import userdata
    _val = userdata.get("ANTHROPIC_API_KEY")
    if _val:                          # None / "" でなければ採用
        ANTHROPIC_API_KEY = _val
        _colab_secrets_ok = True
        print("  [source] Colab Secrets ✅")
    else:
        print("  [warn]  userdata.get() が None/空を返しました")
        print("          → シークレットのトグルが ON になっているか確認してセルを再実行してください")
except Exception as _e:
    print(f"  [warn]  Colab userdata アクセスエラー: {_e}")

# ② フォールバック: 環境変数
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    if ANTHROPIC_API_KEY:
        print("  [source] 環境変数 ANTHROPIC_API_KEY ✅")

# ③ 最終手段: 直接貼り付け（非推奨。git に commit しないよう注意）
# ANTHROPIC_API_KEY = "sk-ant-xxxxxxxx"

# BIS API KEY（同様に Secrets → フォールバック）
try:
    from google.colab import userdata as _ud
    BIS_API_KEY = _ud.get("BIS_API_KEY") or "DEMO_KEY"
except Exception:
    BIS_API_KEY = os.environ.get("BIS_API_KEY", "DEMO_KEY")

# ── 結果表示 ──────────────────────────────────────────────────────────────────
print()
if ANTHROPIC_API_KEY:
    print(f"✅ ANTHROPIC_API_KEY: {'*' * (len(ANTHROPIC_API_KEY)-4)}{ANTHROPIC_API_KEY[-4:]}")
else:
    print("⚠️  ANTHROPIC_API_KEY が取得できませんでした")
    print()
    print("【対処法】")
    print("  1. 左サイドバーの 🔑 アイコン → 「シークレット」を開く")
    print("  2. ANTHROPIC_API_KEY が登録されているか確認")
    print("  3. 「ノートブックのアクセス」列のトグルが 🔵 ON になっているか確認")
    print("  4. トグルを ON にしたら、このセルを再実行（▶ ボタン）")
    print()
    print("  ※ それでも解決しない場合は ③ の直接貼り付けを一時的に使用")

print(f"✅ BIS_API_KEY: {'DEMO_KEY (無料枠)' if BIS_API_KEY == 'DEMO_KEY' else '***設定済み'}")

In [ ]:
# ── パス設定 ──────────────────────────────────────────────────────────────────
from pathlib import Path

BASE           = Path(REPO_BASE)
DATA_DIR       = BASE / "data"
STAGING_DIR    = DATA_DIR / "staging"
SOURCE_DIR     = DATA_DIR / "source"
UNIFIED_DIR    = DATA_DIR / "unified"

# staging サブディレクトリを確保
for sub in ("sanctions", "fefta", "eccn", "patents", "mappings", "faiss"):
    (STAGING_DIR / sub).mkdir(parents=True, exist_ok=True)

print("パス設定完了:")
for name, p in [("BASE", BASE), ("STAGING_DIR", STAGING_DIR), ("UNIFIED_DIR", UNIFIED_DIR)]:
    print(f"  {name}: {p}  ({'OK' if p.exists() else 'NOT FOUND'})")

## ✅ セットアップ完了
次のノートブック → `01_collect_raw_data.ipynb`